# Microglia Handcrafted Localization

This notebook contains the code used for executing and evaluating microglia localization using blob detection with laplacian of gaussians.

## Full Pipeline for Bounding Box Localization

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))
from scripts.filters import *
from scripts.utils import *
from skimage.feature import blob_log
import cv2
import numpy as np
import pyvips
from skimage.segmentation import watershed
from scipy import ndimage as ndi
from tqdm import tqdm


data_dir = None
files = [f for f in Path(data_dir).glob("./*.tiff")]


def binarize_HSV_2(img_np):
    """Binarize image patch for branch extraction and bounding box calculation"""

    # centroid calculations
    img_processed = pre_process_binarize(img_np)
    blobs = blob_log(
        img_processed, min_sigma=20, max_sigma=50, num_sigma=10, threshold=0.2
    )
    centroid = [(y, x) for y, x, r in blobs]

    # HSV color segmentation:
    img_hsv = cv2.cvtColor(img_np, cv2.COLOR_RGB2HSV)

    lb = np.array([2, 0, 0])
    ub = np.array([15, 230, 235])
    mask = cv2.inRange(img_hsv, lb, ub)

    # morphology to connect branches
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    return cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel), centroid


for file in files:
    img = pyvips.Image.new_from_file(file, access="sequential")
    for processed, patch, (y_patch, x_patch) in tqdm(
        create_patches(img, 512, stride=512 / 2, filter=filter_tissue),
        desc="process patches",
    ):
        masked, centroids = binarize_HSV_2(patch)
        dist = ndi.distance_transform_edt(masked)
        markers = np.zeros(masked.shape, dtype=np.int32)
        for i, (y, x) in enumerate(centroids, start=1):
            markers[int(y), int(x)] = i

        labels = np.array(watershed(-dist, markers, mask=masked))
        labels, segmented_cells = filter_size(labels, len(centroids))
        bounding_boxes = calculate_bb(segmented_cells)
        final_bb = []
        for bb in bounding_boxes:
            x1, y1, x2, y2 = bb
            if (x2 - x1) * (y2 - y1) < 16000 or (x2 - x1) < 25 or (y2 - y1) < 25:
                continue
            else:
                final_bb.append(bb)
        for x, y, x2, y2 in final_bb:
            cell = img.crop(x, y, (x2 - x), (y2 - y))

## Evaluation

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))
from scripts.filters import *
from scripts.evaluate import *
from scripts.utils import *
from tqdm import tqdm
from skimage.feature import blob_log
import pyvips
from scipy import ndimage as ndi
from skimage.segmentation import watershed


ground_truth = read_annotations("./evaluation/annotations/gt_bb.json")
ground_truth_pts = read_annotations_points("./evaluation/annotations/gt_points.json")
data_dir = "./evaluation/test"
files = [f for f in Path(data_dir).glob("./*.png")]


def binarize_HSV_2(img_np):
    """Binarize image patch for branch extraction and bounding box calculation"""

    # centroid calculations
    img_processed = pre_process_binarize(img_np)
    blobs = blob_log(
        img_processed, min_sigma=20, max_sigma=50, num_sigma=10, threshold=0.2
    )
    centroid = [(y, x) for y, x, r in blobs]

    # HSV color segmentation:
    img_hsv = cv2.cvtColor(img_np, cv2.COLOR_RGB2HSV)

    lb = np.array([2, 0, 0])
    ub = np.array([15, 230, 235])
    mask = cv2.inRange(img_hsv, lb, ub)

    # morphology to connect branches
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    return cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel), centroid


# calculate localized soma points and corresponding bounding boxes
preds_dict = {}
preds_dict_pts = {}

for idx, file in tqdm(enumerate(files)):
    img_file = pyvips.Image.new_from_file(file, access="sequential")
    patch = np.ndarray(
        buffer=img_file.write_to_memory(),
        dtype=np.uint8,
        shape=[img_file.height, img_file.width, img_file.bands],
    )
    masked, centroids = binarize_HSV_2(patch)
    dist = ndi.distance_transform_edt(masked)
    markers = np.zeros(masked.shape, dtype=np.int32)
    for i, (y, x) in enumerate(centroids, start=1):
        markers[int(y), int(x)] = i

    labels = np.array(watershed(-dist, markers, mask=masked))
    labels, segmented_cells = filter_size(labels, len(centroids))
    bounding_boxes = calculate_bb(segmented_cells)
    final_bb = []
    for bb in bounding_boxes:
        x1, y1, x2, y2 = bb
        if (x2 - x1) * (y2 - y1) < 16000 or (x2 - x1) < 25 or (y2 - y1) < 25:
            continue
        else:
            final_bb.append(bb)
    preds_dict[file.name] = final_bb
    preds_dict_pts[file.name] = centroids
    # plot_predictions(
    #     img_name=file.name,
    #     gt_points=ground_truth_pts,
    #     gt_boxes_raw=ground_truth,
    #     pred_points=centroids,
    #     pred_boxes=final_bb,
    #     dist_threshold=15,
    #     iou_threshold=0.3,
    #     it=idx,
    #     data_set="test"
    # )


# Run evaluation
bulk_eval(ground_truth, preds_dict, iou_threshold=0.3)
bulk_eval(
    ground_truth_pts,
    preds_dict_pts,
    iou_threshold=0.3,
    dist_threshold=15,
    points=True,
    plot=False,
)

36it [01:25,  2.37s/it]

Bounding box evaluation
Img:TPO_67_TPO.tiff_9.png not found in GT but cells were localized
Img:TPO_66_EV2.tiff_10.png not found in GT but cells were localized
Img:TPO_66_EV.tiff_6.png not found in GT but cells were localized
Evaluation output:
TP:35
FP:12
FN:11
Mean IOU:0.6476434890895965
F1-score: 0.7526881720430109
Pointwise evaluation
Img:TPO_67_TPO.tiff_9.png not found in GT but cells were localized
Img:TPO_66_EV2.tiff_10.png not found in GT but cells were localized
Img:TPO_62_EV.tiff_4.png not found in GT but cells were localized
Img:TPO_63_TPO.tiff_1.png not found in GT but cells were localized
Img:TPO_64_EV.tiff_7.png not found in GT but cells were localized
Img:TPO_66_EV.tiff_6.png not found in GT but cells were localized
Img:TPO_63_TPO.tiff_10.png not found in GT but cells were localized
Img:TPO_64_EV.tiff_5.png not found in GT but cells were localized
Img:TPO_60.tiff_7.png not found in GT but cells were localized
Img:TPO_62_EV2.tiff_9.png not found in GT but cells were locali

(39, 57, 7)